# Simulating a periodic dipole with FMMAX

This tutorial uses the Fourier modal method (FMM), closely related to rigorous coupled-wave analysis (RCWA), to simulate an electric dipole in a periodic unit cell. Periodic boundary conditions in the transverse directions mean that the calculation represents an infinite array of identical dipoles rather than one isolated dipole.

FMM represents both material properties and electromagnetic fields with Fourier harmonics in the periodic plane. We will construct a finite Fourier expansion, solve for the layer modes, couple an embedded dipole to those modes, and calculate directional emitted power and a real-space field profile.

Because the Fourier expansion must be truncated, we will also vary the number of retained terms and examine whether quantities of interest become stable. This convergence check is an essential part of establishing numerical accuracy.

## Simulation geometry

We use the following parameters:

- The unit cell is 1.4 µm by 1.4 µm in the periodic x–y plane and 1.2 µm along z.
- The x and y boundaries are periodic, with the same phase in neighboring cells.
- The structure is open along z, allowing radiation to leave in both directions.
- The background is vacuum, with relative permittivity $\varepsilon_r=1$.
- An x-oriented electric dipole is placed at the center of the unit cell and emits at a wavelength of 0.620 µm.

The transverse periodicity turns this nominally single source into an in-phase periodic array of dipoles.

The periodic x–y lattice is specified by two primitive `LatticeVectors`. For this square unit cell, the vectors point along x and y and both have length `pitch`.

FMMAX uses natural units for length: it does not impose a particular built-in unit such as metres. Any length unit may be used, provided that every dimensionful length—including the lattice pitch, layer thickness, wavelength, source position, and plotting coordinates—is expressed consistently. Here, all lengths are in micrometres (µm).

In [2]:
from fmmax import basis

pitch = 1.4  # micrometres
height = 1.2  # micrometres
primitive_lattice_vectors = basis.LatticeVectors(
    u=pitch * basis.X, v=pitch * basis.Y
)

The lattice defines the available reciprocal-lattice vectors. We next choose a finite set of those vectors to represent the source and fields in Fourier space.

`approximate_num_terms` is the target total number of spatial Fourier harmonics across both periodic dimensions; it is not a number specified separately for x and y. FMMAX constructs a symmetric set around the zero-order harmonic so that positive and negative reciprocal vectors are retained in balanced pairs. Because a complete symmetric boundary must be included, the actual value of `expansion.num_terms` can differ slightly from the requested value.

The truncation rule determines the shape of the retained region in reciprocal space. `CIRCULAR` keeps vectors inside a circle, while `PARALLELOGRAMIC` keeps a complete centered parallelogram of reciprocal-lattice indices. We use circular truncation with a target of 200 terms.

In [3]:
approximate_num_terms = 200
expansion = basis.generate_expansion(
    primitive_lattice_vectors=primitive_lattice_vectors,
    approximate_num_terms=approximate_num_terms,
    truncation=basis.Truncation.CIRCULAR,
)

A periodic calculation also requires an in-plane Bloch wavevector, which sets the phase accumulated when translating from one unit cell to the next. Setting this vector to zero selects the Γ point: neighboring cells have identical phase, so the dipoles in the periodic array are driven in phase.

This notebook evaluates only this single in-plane wavevector. It does not integrate over the Brillouin zone.

In [4]:
import jax.numpy as jnp

in_plane_wavevector = jnp.zeros((2,))

The dipole is localized in the x–y plane, so its periodic spatial delta function contains an infinite Fourier series. `sources.dirac_delta_source()` evaluates the coefficients of that series on the finite expansion selected above. Increasing the expansion size therefore improves the representation of the sharply localized source.

Only `jx` will be excited later, making this an x-oriented electric-current dipole. The source is embedded inside the stack and radiates toward both increasing and decreasing z, so the portions of the structure on both sides of the source must be included when solving for its modal amplitudes.

In [5]:
from fmmax import sources

dipole = sources.dirac_delta_source(
    location=jnp.asarray([[pitch / 2, pitch / 2]]),
    in_plane_wavevector=in_plane_wavevector,
    primitive_lattice_vectors=primitive_lattice_vectors,
    expansion=expansion,
)

To place the dipole at the center along z, we represent the surrounding region as two identical layers, each 0.6 µm thick, with the source at their common interface. Because both layers have the same homogeneous, isotropic material, their eigensolutions are identical and only one layer solve is needed.

The code uses a very small imaginary part in the refractive index, giving `permittivity` the value `(1 + 0.0001j) ** 2`. This weak loss acts as a numerical regularization while remaining close to vacuum. The `(1, 1)` permittivity array identifies a spatially uniform layer, allowing FMMAX to use its homogeneous isotropic-medium eigensolver.

The wavelength is stored as a JAX array. FMMAX operations can also accept additional batch dimensions when calculations over several wavelengths or parameters are needed.

In [6]:
from fmmax import fmm

wavelength = jnp.asarray(0.620)
permittivity = jnp.asarray([[(1.0 + 0.0001j) ** 2]])
formulation = fmm.Formulation.FFT
layer_solve_result = fmm.eigensolve_isotropic_media(
    wavelength=wavelength,
    in_plane_wavevector=in_plane_wavevector,
    primitive_lattice_vectors=primitive_lattice_vectors,
    permittivity=permittivity,
    expansion=expansion,
    formulation=formulation
)

layer_thickness = height / 2.0

A scattering matrix relates the forward- and backward-going modal amplitudes at the boundaries of a layer or stack. Here, the two half-stacks must remain separate so that the dipole can be inserted between them rather than cascading the layers first.

We therefore construct one scattering matrix for the layer before the source and one for the layer after it. The layers are identical in this example, so the second scattering matrix can reuse the first.

In [7]:
from fmmax import scattering

s_matrix_before_source = scattering.stack_s_matrix(
    layer_solve_results=[layer_solve_result],
    layer_thicknesses=[layer_thickness],
)

# The layers are identical
s_matrix_after_source = s_matrix_before_source

`sources.amplitudes_for_source()` converts the dipole-current coefficients into the electromagnetic eigenmode amplitudes that satisfy the source discontinuity and the scattering response on both sides.

In [8]:
(
    _,
    fwd_amplitude_before_start,
    bwd_amplitude_before_end,
    fwd_amplitude_after_start,
    bwd_amplitude_after_end,
    _,
) = sources.amplitudes_for_source(
    jx=dipole,
    jy=jnp.zeros_like(dipole),
    jz=jnp.zeros_like(dipole),
    s_matrix_before_source=s_matrix_before_source,
    s_matrix_after_source=s_matrix_after_source,
)

In FMMAX, forward amplitudes propagate toward increasing z, whereas backward amplitudes propagate toward decreasing z. Thus, `fwd_amplitude_after_start` is the outgoing forward amplitude immediately on the increasing-z side of the source, and `bwd_amplitude_before_end` is the outgoing backward amplitude immediately on its decreasing-z side. This convention is tied to the z coordinate rather than to potentially ambiguous labels such as top and bottom.

These modal coefficients are the natural output of the Fourier-domain calculation. The following sections use them to compute power flux and reconstruct fields on a real-space grid.

## Postprocessing

We will extract two kinds of information from the modal amplitudes:

- the fraction of emitted power directed toward increasing z; and
- the x-component of the electric-field intensity on a selected x–y cross-section.

The transverse real-space reconstruction uses a nominal spacing of 0.01 µm. This sampling controls how densely the truncated Fourier series is displayed; it does not increase the number of Fourier harmonics in the underlying solution.

### Directional extraction efficiency

The directional extraction efficiency is the power emitted toward increasing z divided by the total outward power emitted in both z directions. `fields.directional_poynting_flux()` calculates the signed time-averaged flux contributions associated with the retained modes. Forward flux is positive, while flux toward decreasing z is negative, so the latter is negated when forming a positive emitted-power contribution.

The sums in the next cell run over the retained Fourier modes or diffraction orders at the single in-plane wavevector $k_\parallel=0$. They are not a sum or integral over the Brillouin zone.

Because the dipole is centered between identical vacuum-like regions, the two directions are symmetric. The printed extraction efficiency is therefore `0.5`.

In [9]:
from fmmax import fields

# Compute the Poynting flux before and after the source
fwd_flux, bwd_flux = fields.directional_poynting_flux(
    forward_amplitude=fwd_amplitude_after_start,
    backward_amplitude=bwd_amplitude_before_end,
    layer_solve_result=s_matrix_before_source.end_layer_solve_result,
)

# Sum the flux over the retained modes at the single in-plane wavevector.
forward_emitted_power = jnp.sum(fwd_flux)
backward_emitted_power = -jnp.sum(bwd_flux)
total_emitted_power = forward_emitted_power + backward_emitted_power

# The total extracted power is the forward emitted power.
total_extracted_power = forward_emitted_power

# Calculate the extraction efficiency.
extraction_efficiency = total_extracted_power / total_emitted_power
print(extraction_efficiency)

0.5


### Near-field profile

The solved amplitudes are Fourier-domain quantities. `fields.layer_fields_3d()` propagates those amplitudes through the layer and evaluates the inverse Fourier series on a Cartesian grid, returning the electric field, magnetic field, and corresponding coordinates.

The next cell reconstructs the layer on the increasing-z side of the source, selects the x component of the electric field on the z plane nearest 0.5 µm from the source and plots $|E_x|^2$ across the unit cell. The plotted profile is still band-limited by the retained Fourier expansion: using a denser plotting grid provides smoother sampling but does not add unresolved spatial harmonics.

In [ ]:
import matplotlib.pyplot as plt

# Define mesh size and number of grid points in x and y.
mesh_size = 0.01
xy_n_grid = int(pitch / mesh_size)
z_n_grid = int(height / mesh_size)

# Compute the fields in the Cartesian coordinates.
(ef, hf, (x, y, z)) = fields.layer_fields_3d(
    forward_amplitude_start=fwd_amplitude_after_start,
    backward_amplitude_end=bwd_amplitude_after_end,
    layer_solve_result=s_matrix_before_source.end_layer_solve_result,
    layer_thickness=layer_thickness,
    layer_znum=z_n_grid,
    grid_shape=(xy_n_grid, xy_n_grid),
    num_unit_cells=(1, 1)
)

# Take the cross section nearest z = 0.5 µm.
z_target = 0.5
z_index = int(jnp.argmin(jnp.abs(z - z_target)))
cross_section = ef[0, :, :, z_index, 0]
extent = [-pitch / 2, pitch / 2, -pitch / 2, pitch / 2]
plt.imshow(jnp.abs(cross_section)**2, cmap='hot', interpolation='nearest', extent=extent)
plt.xlabel('x (µm)')
plt.ylabel('y (µm)')
plt.title(f'Electric field (E_x) intensity at z={float(z[z_index]):.2f} µm')
plt.show()

## Checking convergence

Truncating the Fourier expansion introduces numerical error. A convergence study repeats the calculation with progressively more retained terms and checks whether the quantities relevant to the application become stable. Larger expansions generally require more computation time and memory.

The final cell repeats the field calculation for target expansion sizes from 10 to 1000 terms and displays the same cross-section for each case. These images can reveal qualitative changes, but increased sharpness alone does not demonstrate convergence. A quantitative study should also compare values such as emitted power, extraction efficiency, or a field norm evaluated at consistent coordinates, and should require those values to change by less than a chosen tolerance as the expansion grows.

A point dipole contains very high spatial frequencies, so fields close to the source may converge more slowly than integrated power quantities. Also remember that the actual number of retained terms can differ slightly from each target because FMMAX preserves a symmetric truncation.

In [ ]:
from fmmax import basis, sources, fmm, scattering, fields

import matplotlib.pyplot as plt
def plot_cross_section(num_terms: int, ax: plt.Axes) -> None:
    """Helper function to plot the cross section of the electric field intensity.
    
    Args:   
        num_terms: Number of terms in the expansion
        ax: Matplotlib axes object
    """
    # Define simulation parameters
    pitch = 1.4  # micrometres
    height = 1.2  # micrometres
    mesh_size = 0.01
    xy_n_grid = int(pitch / mesh_size)
    z_n_grid = int(height / mesh_size)

    # Set up the simulation
    expansion = basis.generate_expansion(
        primitive_lattice_vectors=primitive_lattice_vectors,
        approximate_num_terms=num_terms,
        truncation=basis.Truncation.CIRCULAR,
    )

    dipole = sources.dirac_delta_source(
        location=jnp.asarray([[pitch / 2, pitch / 2]]),
        in_plane_wavevector=in_plane_wavevector,
        primitive_lattice_vectors=primitive_lattice_vectors,
        expansion=expansion,
    )

    wavelength = jnp.asarray(0.620)
    permittivity = jnp.asarray([[(1.0 + 0.0001j) ** 2]])
    formulation = fmm.Formulation.FFT
    layer_solve_result = fmm.eigensolve_isotropic_media(
        wavelength=wavelength,
        in_plane_wavevector=in_plane_wavevector,
        primitive_lattice_vectors=primitive_lattice_vectors,
        permittivity=permittivity,
        expansion=expansion,
        formulation=formulation
    )

    layer_thickness = height / 2.0

    s_matrix_before_source = scattering.stack_s_matrix(
        layer_solve_results=[layer_solve_result],
        layer_thicknesses=[layer_thickness],
    )

    # The layers are identical
    s_matrix_after_source = s_matrix_before_source

    (
        _,
        fwd_amplitude_before_start,
        bwd_amplitude_before_end,
        fwd_amplitude_after_start,
        bwd_amplitude_after_end,
        _,
    ) = sources.amplitudes_for_source(
        jx=dipole,
        jy=jnp.zeros_like(dipole),
        jz=jnp.zeros_like(dipole),
        s_matrix_before_source=s_matrix_before_source,
        s_matrix_after_source=s_matrix_after_source,
    )

    # Compute the fields in the Cartesian coordinates
    (ef, hf, (x, y, z)) = fields.layer_fields_3d(
        forward_amplitude_start=fwd_amplitude_after_start,
        backward_amplitude_end=bwd_amplitude_after_end,
        layer_solve_result=s_matrix_before_source.end_layer_solve_result,
        layer_thickness=layer_thickness,
        layer_znum=z_n_grid,
        grid_shape=(xy_n_grid, xy_n_grid),
        num_unit_cells=(1, 1)
    )

    # Plot the cross section nearest z = 0.5 µm.
    z_target = 0.5
    z_index = int(jnp.argmin(jnp.abs(z - z_target)))
    cross_section = ef[0, :, :, z_index, 0]
    extent = [-pitch / 2, pitch / 2, -pitch / 2, pitch / 2]
    ax.imshow(jnp.abs(cross_section) ** 2, cmap='hot', interpolation='nearest', extent=extent)
    ax.set_xlabel('x (µm)')
    ax.set_ylabel('y (µm)')
    ax.set_title(
        f'Approx. terms = {num_terms}, z={float(z[z_index]):.2f} µm'
    )

n_term_list = [10, 20, 50, 100, 200, 300, 400, 500, 1000]
fig, axs = plt.subplots(3, 3, figsize=(15, 15))
for i, n_term in enumerate(n_term_list):
    plot_cross_section(n_term, axs.ravel()[i])
plt.tight_layout()
plt.show()
